In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_classic.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_ollama import ChatOllama
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.passthrough import RunnablePassthrough

In [67]:
with open('document.txt', encoding='utf-8') as f:
    text = f.read()
    
headers_to_split_on = [
    ("#", "Header")
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
chunks = markdown_splitter.split_text(text)

In [68]:
embedding_model = HuggingFaceEmbeddings(
    model_name="keepitreal/vietnamese-sbert", 
    model_kwargs={'device': 'cuda', 'local_files_only': True},   
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18763.30it/s]


In [69]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model,
    distance_strategy=DistanceStrategy.COSINE
)

In [70]:
llm = ChatOllama(
    model="qwen2.5:3b",
    num_ctx=1024,
    num_predict=256,
    num_gpu=0,
    temperature=0
)

In [71]:
def prompt_chat(x, y):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(x),
        HumanMessagePromptTemplate.from_template(y)
    ])
    return prompt

In [72]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5
vectorstore_retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k':5}
)
retrievers = EnsembleRetriever(
    retrievers = [bm25_retriever, vectorstore_retriever],
    weights = [0.5, 0.5]
)

In [73]:
system_instruction = (
    "Bạn là một Chatbot tư vấn y tế thông minh và thân thiện.\n"
    "Nhiệm vụ của bạn là đọc thông tin được cung cấp để diễn đạt lại thành câu trả lời tự nhiên cho người dùng về vấn đề họ gặp phải.\n\n"
    
    "QUY TẮC PHẢN HỒI:\n"
    "- Sử dụng ngôn ngữ ân cần, mạch lạc giống một chuyên viên y tế (ví dụ: 'Qua các triệu chứng bạn chia sẻ, hệ thống nhận thấy...').\n"
    "- Không thêm, không bớt, không suy luận, không nói rằng không tìm thấy dữ liệu.\n"
    "- Triển khai mỗi ý nằm ổ một dòng, không đưa câu trả lời lên một dòng duy nhất.\n"
    "- Luôn kết thúc bằng một lời khuyên người dùng nên đến các cơ sở y tế hoặc gặp bác sĩ để thăm khám trực tiếp, vì đây chỉ là thông tin tham khảo từ hệ thống."
)

human_instruction = (
    "Báo cáo kết quả đối chiếu từ hệ thống Python:\n{context}\n\n"
    "Câu hỏi gốc của người dùng: {question}"
)

In [74]:
docs = retrievers.invoke('tê bì chân tay có triệu chứng gì')

In [75]:
def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

In [76]:
print(docs)

[Document(metadata={'Header': 'Tổng quan bệnh Tê bì tay chân'}, page_content='# Tổng quan bệnh Tê bì tay chân  \nTê bì chân tay là gì?\nTay chân bình thường sẽ dựa vào cảm giác để điều chỉnh những hoạt động như rút tay chân lại khi chạm phải vật nóng, điều chỉnh khi địa hình thay đổi. Nếu bạn bị tê tay chân thì sẽ gây giảm cảm giác và thậm chí năng hơn có thể gây mất cảm giác hoàn toàn.\nTình trạng này có thể khởi phát rất nhẹ nhàng như tê rần ở các đầu ngón tay, có cảm giác như bị châm chích ở đầu ngón tay hoặc bị giảm cảm giác. Những triệu chứng này có thể ngày càng nặng hơn, lan dần lên phía bàn tay, cổ tay, cẳng tay, cánh tay… và có thể đi đến tình trạng mất hết cảm giác. Tê bì chân tay khi mang thai, tê bì chân tay ở người già là tình trạng rất phổ biến trong cộng đồng.\nTình trạng có thể là tạm thời hoặc là triệu chứng của bệnh lý khác.'), Document(metadata={'Header': 'Triệu chứng bệnh Tê bì tay chân'}, page_content='# Triệu chứng bệnh Tê bì tay chân  \nĐau mỏi cổ vai gáy lan xuố

In [77]:
rag_chain = (
    {'context':lambda _: format_docs(docs), 'question': RunnablePassthrough()}
    | prompt_chat(system_instruction, human_instruction)
    | llm
    | StrOutputParser()
)

In [78]:
answer = rag_chain.invoke(docs)

In [79]:
print(answer)

Dựa trên thông tin bạn cung cấp, tôi sẽ tổng hợp lại các thông tin quan trọng về triệu chứng và biến chứng của thoát vị đĩa đệm như sau:

1. Triệu chứng:
- Đau, tê bì, yếu cơ ngày càng nặng
- Tần suất đi tiểu hoặc táo bón
- Mất cảm giác ở các vùng như bắp đùi trong, phía sau chân, vùng quanh hậu môn
- Rối loạn cơ vòng: bí tiểu, đái dầm dề

2. Biến chứng nếu không điều trị:
- Liệt nửa người hoặc bại liệt cả người
- Hội chứng đuôi ngựa: táo bón, đái dầm dề
- Cơ teo, suy yếu, giảm khả năng vận động
- Hẹp khoang sống, chèn ép rễ thần kinh
- Rối loạn chức năng đi tiểu

Để tránh những biến chứng nghiêm trọng, bệnh nhân cần đến bệnh viện và tham khảo ý kiến bác sĩ ngay khi có những biểu hiện trên. Điều trị sớm sẽ giúp giảm nguy cơ biến chứng và cải thiện chất lượng cuộc sống.
